In [1]:
# =========================
# 环境初始化
# =========================

import os

print("RAG环境初始化完成")

# =========================
# 当前路径检查
# =========================

import os

print(
    os.getcwd()
)

RAG环境初始化完成
D:\Anaconda\Project\AI-Coding-Agent\agent-learning


In [2]:
# =========================
# 文档加载
# =========================

from pathlib import Path
from langchain_community.document_loaders import TextLoader

file_path = Path(
    "day03/knowledge/Unity.md"
)

loader = TextLoader(
    str(file_path),
    encoding="utf-8"
)

documents = loader.load()

print("文档数量:",len(documents))

print(documents[0].page_content[:200])

C:\Users\admin\AppData\Local\Temp\ipykernel_26788\3793577182.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


文档数量: 1
# Unity开发规范


## 项目架构

Unity项目通常采用模块化设计。


主要模块：

- Manager管理模块
- Data数据模块
- UI交互模块
- Controller控制模块


## MonoBehaviour

MonoBehaviour是Unity脚本基础类。

常用于：

- 生命周期管理
- 游戏对象控制
- 事件处理


## ScriptableObject


In [28]:
# =========================
# 批量加载知识库
# =========================

from pathlib import Path
from langchain_community.document_loaders import TextLoader

documents = []

knowledge_path = Path("day03/knowledge")

# 遍历知识库目录中的Markdown文件
for file in knowledge_path.glob("*.md"):

    loader = TextLoader(
        str(file),
        encoding="utf-8"
    )

    # 加载文档并加入知识库列表
    documents.extend(loader.load())

print("知识文档数量:", len(documents))

知识文档数量: 2


In [29]:
# =========================
# 查看知识来源
# =========================

for doc in documents:
    print("来源:", doc.metadata["source"])

来源: day03\knowledge\Inventory.md
来源: day03\knowledge\Unity.md


In [31]:
# =========================
# 文档切片
# =========================

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("切片数量:", len(chunks))

切片数量: 13


In [32]:
# =========================
# 查看Chunk内容
# =========================

for i, chunk in enumerate(chunks):
    print("Chunk:", i)
    print(chunk.page_content[:100])
    print("来源:", chunk.metadata["source"])
    print("----------------")

Chunk: 0
# Unity背包系统


## InventoryManager

InventoryManager负责玩家背包管理。


主要功能：

- 添加物品
- 删除物品
- 查询物品
- 保存背包
来源: day03\knowledge\Inventory.md
----------------
Chunk: 1
主要功能：

- 添加物品
- 删除物品
- 查询物品
- 保存背包


## ItemDefinition

ItemDefinition定义物品数据。


包含：
来源: day03\knowledge\Inventory.md
----------------
Chunk: 2
## ItemDefinition

ItemDefinition定义物品数据。


包含：

- ID
- 名称
- 图标
- 类型
- 最大堆叠数量


## InventorySlot
来源: day03\knowledge\Inventory.md
----------------
Chunk: 3
- ID
- 名称
- 图标
- 类型
- 最大堆叠数量


## InventorySlot

InventorySlot表示背包格子。


包含：

- Item
- Count
来源: day03\knowledge\Inventory.md
----------------
Chunk: 4
InventorySlot表示背包格子。


包含：

- Item
- Count

## 数据保存

InventoryManager通常不会直接保存MonoBehaviour状态。
来源: day03\knowledge\Inventory.md
----------------
Chunk: 5
InventoryManager通常不会直接保存MonoBehaviour状态。

推荐使用：

- JSON
- SQLite
- ScriptableObject

进行数据持久化。
来源: day03\knowledge\Inventory.md
----------------
Chunk: 6
- JSON
- SQLite
- ScriptableObject

进行数据持久化。


## 网络同步

多人游戏中：

Inventory数据需要同步

In [49]:
# =========================
# 当前路径检查
# =========================

import os

print("当前路径:", os.getcwd())
print("当前目录:", os.listdir())

当前路径: D:\Anaconda\Project\AI-Coding-Agent\agent-learning
当前目录: ['.env', '.ipynb_checkpoints', 'day01', 'day02', 'day03', 'models']


In [37]:
# ==========================================
# 从本地目录加载 Embedding 模型
# ==========================================

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="./models/bge-small-zh-v1.5",
    model_kwargs={"local_files_only": True}
)

print("本地Embedding模型加载完成")

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 7903.28it/s]

本地Embedding模型加载完成


In [38]:
# ==========================================
# 检查模型权重文件大小
# ==========================================

from pathlib import Path

model_path = Path("./models/bge-small-zh-v1.5")
weight_path = model_path / "model.safetensors"

print(f"模型目录存在：{model_path.exists()}")
print(f"权重文件存在：{weight_path.exists()}")
print(f"权重文件大小：{weight_path.stat().st_size / 1024 / 1024:.2f} MB")

模型目录存在：True
权重文件存在：True
权重文件大小：91.39 MB


In [39]:
# ==============================
# 查看当前 Embedding 模型配置
# ==============================

print(f"当前Embedding模型：{embedding_model.model_name}")

当前Embedding模型：./models/bge-small-zh-v1.5


In [40]:
# =========================
# 测试文本向量
# =========================

vector = embedding_model.embed_query(
    "Unity背包系统如何设计"
)

print("向量维度:", len(vector))

向量维度: 512


In [34]:
# =========================
# 创建FAISS向量库
# =========================

from langchain_community.vectorstores import FAISS


vector_store = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS向量库创建完成")

FAISS向量库创建完成


In [35]:
# =========================
# 保存FAISS索引
# =========================

vector_store.save_local(
    "day03/vector_store"
)

print("FAISS索引保存完成")

FAISS索引保存完成


In [41]:
# =========================
# RAG检索工具
# =========================

def retrieve_knowledge(query, k=3):
    """
    根据问题检索知识库
    """

    results = vector_store.similarity_search(
        query,
        k=k
    )

    context = ""

    for doc in results:
        context += "\n来源:" + doc.metadata["source"]
        context += "\n" + doc.page_content

    return context


print("RAG检索工具创建完成")

RAG检索工具创建完成


In [43]:
# =========================
# 测试RAG检索工具
# =========================

context = retrieve_knowledge(
    "Unity背包如何管理物品"
)

print(context)


来源:day03\knowledge\Inventory.md
# Unity背包系统


## InventoryManager

InventoryManager负责玩家背包管理。


主要功能：

- 添加物品
- 删除物品
- 查询物品
- 保存背包
来源:day03\knowledge\Inventory.md
主要功能：

- 添加物品
- 删除物品
- 查询物品
- 保存背包


## ItemDefinition

ItemDefinition定义物品数据。


包含：
来源:day03\knowledge\Inventory.md
InventorySlot表示背包格子。


包含：

- Item
- Count

## 数据保存

InventoryManager通常不会直接保存MonoBehaviour状态。


In [46]:
# =========================
# DeepSeek配置
# =========================

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

print("DeepSeek连接完成")

DeepSeek连接完成


In [47]:
# =========================
# RAG Agent
# =========================

def rag_agent(state):

    print("[RAG Agent]开始执行")

    question = state["task"]

    context = retrieve_knowledge(
        question
    )

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content":
                """
你是一名Unity技术专家。
根据提供的知识库回答问题。
如果知识库没有相关内容，不要编造。
"""
            },
            {
                "role": "user",
                "content":
                f"""
问题:

{question}


知识库:

{context}
"""
            }
        ]
    )

    state["rag_context"] = response.choices[0].message.content

    print("[RAG Agent]执行完成")

    return state

In [48]:
state = {
    "task":"设计Unity背包系统"
}

result = rag_agent(state)

print(result["rag_context"])

[RAG Agent]开始执行
[RAG Agent]执行完成
基于知识库内容，我为您设计一个Unity背包系统的基础架构：

## Unity背包系统设计

### 1. 数据模块 (Data)

```csharp
// 物品数据结构
[System.Serializable]
public class Item
{
    public int ID;          // 物品ID
    public string Name;     // 物品名称
    public Sprite Icon;     // 物品图标
    public ItemType Type;   // 物品类型
    public int MaxStack;    // 最大堆叠数量
}

// 物品类型枚举
public enum ItemType
{
    Weapon,
    Armor,
    Potion,
    Material,
    Quest
}
```

### 2. 背包格子 (InventorySlot)

```csharp
// 背包格子类
[System.Serializable]
public class InventorySlot
{
    public Item item;       // 存储的物品
    public int Count;       // 物品数量

    public bool IsEmpty => item == null || Count == 0;
}
```

### 3. 管理器模块 (InventoryManager)

```csharp
// 背包管理器
public class InventoryManager : MonoBehaviour
{
    // 设置背包容量
    public int inventorySize = 20;
    
    // 背包格子列表
    private List<InventorySlot> inventorySlots;

    void Start()
    {
        InitializeInventory();
    }

    // 初始化背包
    void InitializeInventory